# Practico Parcial 1 - Decision optima y probabilidad de error en canal AWGN

Este notebook desarrolla el ejercicio de decision binaria en un canal AWGN, con todos los despejes matematicos escritos paso a paso.

Datos del problema:

$$P_H(0)=P_H(1)=\frac{1}{2}$$

Bajo la hipotesis $H=0$:

$$Y=-1+Z$$

Bajo la hipotesis $H=1$:

$$Y=2+Z$$

donde:

$$Z\sim N(0,2)$$

Interpretamos $N(0,2)$ como una distribucion normal con media $0$ y varianza $2$:

$$\mu_Z=0$$

$$\sigma_Z^2=2$$

$$\sigma_Z=\sqrt{2}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf

plt.style.use("seaborn-v0_8-whitegrid")

mu0 = -1
mu1 = 2
sigma2 = 2
sigma = np.sqrt(sigma2)

p_h0 = 0.5
p_h1 = 0.5

rng = np.random.default_rng(20260508)

## 1. Densidad de la variable de ruido $Z$

Si una variable aleatoria gaussiana cumple:

$$X\sim N(\mu,\sigma^2)$$

su funcion de densidad de probabilidad es:

$$f_X(x)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right]$$

En este ejercicio:

$$Z\sim N(0,2)$$

por lo tanto:

$$\mu_Z=0$$

$$\sigma_Z^2=2$$

Reemplazamos en la formula general:

$$f_Z(z)=\frac{1}{\sqrt{2\pi\cdot 2}}\exp\left[-\frac{(z-0)^2}{2\cdot 2}\right]$$

Simplificamos el denominador de la constante:

$$\sqrt{2\pi\cdot 2}=\sqrt{4\pi}$$

Simplificamos el denominador del exponente:

$$2\cdot 2=4$$

Entonces:

$$\boxed{f_Z(z)=\frac{1}{\sqrt{4\pi}}\exp\left(-\frac{z^2}{4}\right)}$$


## 2. Densidad condicional $f_{Y|H}(y|0)$

Bajo la hipotesis $H=0$ se cumple:

$$Y=-1+Z$$

Queremos expresar la densidad de $Y$ cuando $H=0$.

Primero despejamos $Z$ en funcion de $Y$:

$$Y=-1+Z$$

Sumamos $1$ a ambos lados:

$$Y+1=Z$$

Entonces:

$$Z=Y+1$$

Si observamos un valor particular $Y=y$, entonces:

$$z=y+1$$

Como el desplazamiento es lineal y la derivada vale $1$, la densidad queda:

$$f_{Y|H}(y|0)=f_Z(y+1)$$

Usamos la densidad de $Z$:

$$f_Z(z)=\frac{1}{\sqrt{4\pi}}\exp\left(-\frac{z^2}{4}\right)$$

Reemplazamos $z$ por $y+1$:

$$f_{Y|H}(y|0)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y+1)^2}{4}\right]$$

Por lo tanto:

$$\boxed{f_{Y|H}(y|0)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y+1)^2}{4}\right]}$$

Equivalentemente:

$$Y|H=0\sim N(-1,2)$$


## 3. Densidad condicional $f_{Y|H}(y|1)$

Bajo la hipotesis $H=1$ se cumple:

$$Y=2+Z$$

Primero despejamos $Z$:

$$Y=2+Z$$

Restamos $2$ a ambos lados:

$$Y-2=Z$$

Entonces:

$$Z=Y-2$$

Si observamos un valor particular $Y=y$, entonces:

$$z=y-2$$

La densidad condicional se obtiene evaluando la densidad de $Z$ en $y-2$:

$$f_{Y|H}(y|1)=f_Z(y-2)$$

Usamos:

$$f_Z(z)=\frac{1}{\sqrt{4\pi}}\exp\left(-\frac{z^2}{4}\right)$$

Reemplazamos $z$ por $y-2$:

$$f_{Y|H}(y|1)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y-2)^2}{4}\right]$$

Por lo tanto:

$$\boxed{f_{Y|H}(y|1)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y-2)^2}{4}\right]}$$

Equivalentemente:

$$Y|H=1\sim N(2,2)$$


In [ ]:
def gaussian_pdf(y, mu, sigma2):
    return (1 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-((y - mu) ** 2) / (2 * sigma2))

def f_y_given_h0(y):
    return gaussian_pdf(y, mu0, sigma2)

def f_y_given_h1(y):
    return gaussian_pdf(y, mu1, sigma2)

y_values = np.linspace(-7, 8, 800)
pdf_h0 = f_y_given_h0(y_values)
pdf_h1 = f_y_given_h1(y_values)

plt.figure(figsize=(10, 5))
plt.plot(y_values, pdf_h0, label=r"$f_{Y|H}(y|0)$, media $-1$", linewidth=2)
plt.plot(y_values, pdf_h1, label=r"$f_{Y|H}(y|1)$, media $2$", linewidth=2)
plt.xlabel("y")
plt.ylabel("Densidad")
plt.title("Funciones de densidad condicionadas")
plt.legend()
plt.show()

## 4. Nivel de decision optimo cualitativo

Las dos densidades son gaussianas con la misma varianza:

$$\sigma_0^2=\sigma_1^2=2$$

Sus medias son:

$$\mu_0=-1$$

$$\mu_1=2$$

Como las hipotesis son igualmente probables:

$$P_H(0)=P_H(1)=\frac{1}{2}$$

la regla MAP se reduce a la regla ML.

Cualitativamente, el nivel de decision optimo esta donde se cruzan las dos densidades. Como ambas tienen la misma varianza, ese cruce queda en el punto medio entre las medias:

$$\gamma=\frac{\mu_0+\mu_1}{2}$$

$$\gamma=\frac{-1+2}{2}$$

$$\gamma=\frac{1}{2}$$

Entonces, cualitativamente:

$$\boxed{\gamma=0.5}$$

La decision es optima bajo la regla **ML**, porque las hipotesis son equiprobables. Tambien puede decirse que coincide con la regla **MAP**, que es la que minimiza la probabilidad de error cuando los costos de error son iguales.


## 5. Nivel de decision optimo cuantitativo

La regla ML decide $H=1$ si la observacion $y$ es mas verosimil bajo $H=1$ que bajo $H=0$:

$$\hat H=1 \quad \text{si} \quad f_{Y|H}(y|1)\ge f_{Y|H}(y|0)$$

Sustituimos las densidades:

$$\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y-2)^2}{4}\right] \ge \frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y+1)^2}{4}\right]$$

La constante $\frac{1}{\sqrt{4\pi}}$ es positiva y aparece en ambos lados, entonces se cancela:

$$\exp\left[-\frac{(y-2)^2}{4}\right] \ge \exp\left[-\frac{(y+1)^2}{4}\right]$$

Aplicamos logaritmo natural a ambos lados. Como $\ln(x)$ es creciente, no cambia el sentido de la desigualdad:

$$\ln\left(\exp\left[-\frac{(y-2)^2}{4}\right]\right) \ge \ln\left(\exp\left[-\frac{(y+1)^2}{4}\right]\right)$$

Usamos que $\ln(e^a)=a$:

$$-\frac{(y-2)^2}{4} \ge -\frac{(y+1)^2}{4}$$

Multiplicamos ambos lados por $4$, que es positivo, por lo tanto no cambia el sentido:

$$-(y-2)^2 \ge -(y+1)^2$$

Multiplicamos ambos lados por $-1$. Como es un numero negativo, cambia el sentido de la desigualdad:

$$(y-2)^2 \le (y+1)^2$$

Desarrollamos ambos cuadrados:

$$(y-2)^2=y^2-4y+4$$

$$(y+1)^2=y^2+2y+1$$

Reemplazamos:

$$y^2-4y+4 \le y^2+2y+1$$

Restamos $y^2$ en ambos lados:

$$-4y+4 \le 2y+1$$

Restamos $1$ en ambos lados:

$$-4y+3 \le 2y$$

Sumamos $4y$ en ambos lados:

$$3 \le 6y$$

Dividimos por $6$, que es positivo:

$$\frac{3}{6}\le y$$

Simplificamos:

$$\frac{1}{2}\le y$$

Entonces la regla queda:

$$\hat H=1 \quad \text{si} \quad y\ge \frac{1}{2}$$

$$\hat H=0 \quad \text{si} \quad y<\frac{1}{2}$$

Por lo tanto:

$$\boxed{\gamma=\frac{1}{2}=0.5}$$


In [ ]:
gamma = (mu0 + mu1) / 2

plt.figure(figsize=(10, 5))
plt.plot(y_values, pdf_h0, label=r"$f_{Y|H}(y|0)$", linewidth=2)
plt.plot(y_values, pdf_h1, label=r"$f_{Y|H}(y|1)$", linewidth=2)
plt.axvline(gamma, color="black", linestyle="--", label=fr"$\gamma={gamma:.1f}$")
plt.fill_between(y_values, 0, pdf_h0, where=(y_values >= gamma), alpha=0.25, label="Error si H=0")
plt.fill_between(y_values, 0, pdf_h1, where=(y_values < gamma), alpha=0.25, label="Error si H=1")
plt.xlabel("y")
plt.ylabel("Densidad")
plt.title("Nivel de decision optimo y regiones de error")
plt.legend()
plt.show()

print(f"Nivel de decision optimo gamma = {gamma:.4f}")

## 6. Probabilidad de error en terminos de la funcion Q

La regla de decision es:

$$\hat H=1 \quad \text{si} \quad Y\ge \gamma$$

$$\hat H=0 \quad \text{si} \quad Y<\gamma$$

con:

$$\gamma=\frac{1}{2}$$

La probabilidad de error total es:

$$P_e=P_H(0)P(\hat H=1|H=0)+P_H(1)P(\hat H=0|H=1)$$

Como las hipotesis son equiprobables:

$$P_H(0)=P_H(1)=\frac{1}{2}$$

queda:

$$P_e=\frac{1}{2}P(\hat H=1|H=0)+\frac{1}{2}P(\hat H=0|H=1)$$

Reemplazamos la regla de decision:

$$P_e=\frac{1}{2}P(Y\ge \gamma|H=0)+\frac{1}{2}P(Y<\gamma|H=1)$$


## 7. Primer termino de error: decidir $H=1$ cuando se transmitio $H=0$

Bajo $H=0$:

$$Y=-1+Z$$

Queremos calcular:

$$P(Y\ge \gamma|H=0)$$

Reemplazamos $Y$:

$$P(-1+Z\ge \gamma)$$

Sumamos $1$ dentro de la desigualdad:

$$P(Z\ge \gamma+1)$$

Ahora reemplazamos $\gamma=\frac{1}{2}$:

$$P\left(Z\ge \frac{1}{2}+1\right)$$

$$P\left(Z\ge \frac{3}{2}\right)$$

Como $Z\sim N(0,2)$, estandarizamos dividiendo por $\sigma=\sqrt{2}$:

$$P\left(\frac{Z}{\sqrt{2}}\ge \frac{\frac{3}{2}}{\sqrt{2}}\right)$$

Definimos:

$$X=\frac{Z}{\sqrt{2}}$$

Entonces:

$$X\sim N(0,1)$$

Por definicion de la funcion Q:

$$Q(a)=P(X\ge a), \qquad X\sim N(0,1)$$

Por lo tanto:

$$P(Y\ge \gamma|H=0)=Q\left(\frac{3}{2\sqrt{2}}\right)$$


## 8. Segundo termino de error: decidir $H=0$ cuando se transmitio $H=1$

Bajo $H=1$:

$$Y=2+Z$$

Queremos calcular:

$$P(Y<\gamma|H=1)$$

Reemplazamos $Y$:

$$P(2+Z<\gamma)$$

Restamos $2$ dentro de la desigualdad:

$$P(Z<\gamma-2)$$

Reemplazamos $\gamma=\frac{1}{2}$:

$$P\left(Z<\frac{1}{2}-2\right)$$

$$P\left(Z<-\frac{3}{2}\right)$$

Estandarizamos dividiendo por $\sqrt{2}$:

$$P\left(\frac{Z}{\sqrt{2}}<-\frac{\frac{3}{2}}{\sqrt{2}}\right)$$

Con:

$$X=\frac{Z}{\sqrt{2}}\sim N(0,1)$$

queda:

$$P\left(X<-\frac{3}{2\sqrt{2}}\right)$$

Por simetria de la normal estandar:

$$P(X<-a)=P(X>a)=Q(a)$$

Entonces:

$$P(Y<\gamma|H=1)=Q\left(\frac{3}{2\sqrt{2}}\right)$$


## 9. Resultado final de $P_e$

Teniamos:

$$P_e=\frac{1}{2}P(Y\ge \gamma|H=0)+\frac{1}{2}P(Y<\gamma|H=1)$$

Ya obtuvimos:

$$P(Y\ge \gamma|H=0)=Q\left(\frac{3}{2\sqrt{2}}\right)$$

$$P(Y<\gamma|H=1)=Q\left(\frac{3}{2\sqrt{2}}\right)$$

Reemplazamos:

$$P_e=\frac{1}{2}Q\left(\frac{3}{2\sqrt{2}}\right)+\frac{1}{2}Q\left(\frac{3}{2\sqrt{2}}\right)$$

Sacamos factor comun:

$$P_e=\left(\frac{1}{2}+\frac{1}{2}\right)Q\left(\frac{3}{2\sqrt{2}}\right)$$

$$P_e=Q\left(\frac{3}{2\sqrt{2}}\right)$$

Por lo tanto:

$$\boxed{P_e=Q\left(\frac{3}{2\sqrt{2}}\right)}$$

Como $\frac{3}{2\sqrt{2}}\approx 1.0607$, tambien se puede escribir:

$$\boxed{P_e=Q(1.0607)}$$


In [ ]:
def Q_function(x):
    return 0.5 * (1 - erf(x / np.sqrt(2)))

q_argument = 3 / (2 * np.sqrt(2))
pe_q = Q_function(q_argument)

print(f"Argumento de Q = 3 / (2*sqrt(2)) = {q_argument:.6f}")
print(f"Pe = Q({q_argument:.6f}) = {pe_q:.6f}")

## 10. Verificacion Monte Carlo

La simulacion Monte Carlo no reemplaza el desarrollo teorico, pero ayuda a comprobar el resultado.

Procedimiento:

1. Se generan hipotesis equiprobables $H=0$ y $H=1$.
2. Se genera ruido $Z\sim N(0,2)$.
3. Si $H=0$, se genera $Y=-1+Z$.
4. Si $H=1$, se genera $Y=2+Z$.
5. Se decide con el umbral $\gamma=0.5$.
6. Se estima $P_e$ como la frecuencia relativa de errores.


In [ ]:
nb_samples = 500_000

h = rng.integers(0, 2, size=nb_samples)
z = rng.normal(loc=0, scale=sigma, size=nb_samples)
y = np.where(h == 0, mu0 + z, mu1 + z)

h_hat = np.where(y >= gamma, 1, 0)
pe_mc = np.mean(h_hat != h)

print(f"Pe teorica = {pe_q:.6f}")
print(f"Pe Monte Carlo = {pe_mc:.6f}")
print(f"Diferencia absoluta = {abs(pe_mc - pe_q):.6f}")

## 11. Respuesta final

Las densidades condicionales son:

$$\boxed{f_{Y|H}(y|0)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y+1)^2}{4}\right]}$$

$$\boxed{f_{Y|H}(y|1)=\frac{1}{\sqrt{4\pi}}\exp\left[-\frac{(y-2)^2}{4}\right]}$$

El nivel de decision optimo es:

$$\boxed{\gamma=0.5}$$

La regla de decision es:

$$\boxed{\hat H=1 \text{ si } y\ge 0.5}$$

$$\boxed{\hat H=0 \text{ si } y<0.5}$$

Como las hipotesis son equiprobables, la decision optima se obtiene con ML, y coincide con MAP.

La probabilidad de error es:

$$\boxed{P_e=Q\left(\frac{3}{2\sqrt{2}}\right)}$$

Numericamente:

$$\boxed{P_e\approx 0.1444}$$
